In [92]:
import torch
import torch.nn as nn

In [93]:
num_tokens = 6
embed_dim = 4
batch_num = 2
x1 = torch.rand(num_tokens,embed_dim)
x2 = torch.rand(num_tokens,embed_dim)
batch = torch.stack((x1,x2), dim=0)
print(batch, "\n batch", batch.shape)

tensor([[[0.9556, 0.5038, 0.8984, 0.6663],
         [0.2130, 0.1277, 0.5406, 0.5112],
         [0.3903, 0.8507, 0.3892, 0.1918],
         [0.0201, 0.5711, 0.9077, 0.4508],
         [0.6841, 0.1498, 0.5564, 0.6475],
         [0.7013, 0.1575, 0.7286, 0.4230]],

        [[0.0531, 0.7238, 0.5847, 0.0742],
         [0.9952, 0.0745, 0.4327, 0.8838],
         [0.9544, 0.8018, 0.8521, 0.7725],
         [0.0158, 0.4791, 0.1315, 0.2001],
         [0.4165, 0.7678, 0.5357, 0.8329],
         [0.0727, 0.2200, 0.1689, 0.4384]]]) 
 batch torch.Size([2, 6, 4])


In [94]:
keyWeightFunc = nn.Linear(in_features=embed_dim, out_features=embed_dim, bias=False)
queryWeightFunc = nn.Linear(in_features=embed_dim, out_features=embed_dim, bias=False)
valueWeightFunc = nn.Linear(in_features=embed_dim, out_features=embed_dim, bias=False)
key = keyWeightFunc(batch)
query = queryWeightFunc(batch)
value = valueWeightFunc(batch)
print("key",key.shape)
print("key-->",key.shape[0], "input", key.shape[1], "tokens",key.shape[2], "embed_dim")

key torch.Size([2, 6, 4])
key--> 2 input 6 tokens 4 embed_dim


In [95]:
num_heads = 2
head_dim = embed_dim//num_heads
keyView = key.view(batch_num, num_tokens, num_heads, head_dim)
queryView = query.view(batch_num, num_tokens, num_heads, head_dim)
valueView = value.view(batch_num, num_tokens, num_heads, head_dim)
print("keyView",keyView.shape)
print("keyView-->",keyView.shape[0],"inputs", keyView.shape[1], "tokens", keyView.shape[2], "no.of heads", keyView.shape[3],"head dimesion")
#head_dim cannot be a float. it needs to be a proper integer. so, set accordingly

keyView torch.Size([2, 6, 2, 2])
keyView--> 2 inputs 6 tokens 2 no.of heads 2 head dimesion


In [96]:
key = keyView.transpose(1,2)
query = queryView.transpose(1,2)
value = valueView.transpose(1,2)
print("after transposing Key",key.shape)
print(key.shape[0],"inputs", key.shape[1],"no.of heads", key.shape[2], "tokens", key.shape[3],"head dimesion")
#why are we performing transpose here?
#We are transforming into 2 inputs -> 2 heads -> token representation (6x2)
# so when multiplication happens, it is carried out between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads

after transposing Key torch.Size([2, 2, 6, 2])
2 inputs 2 no.of heads 6 tokens 2 head dimesion


In [97]:
keyTranspose = key.transpose(2,3)
print(keyTranspose.shape)
print(keyTranspose.shape[0],"inputs", keyTranspose.shape[1],"no.of heads", keyTranspose.shape[2], "head dimesion", keyTranspose.shape[3],"tokens")
attentionScore = query @ keyTranspose
#for 4 dimension matmul 
#Tensor A shape: (..., M, K)Tensor B shape: (..., K, N) The size of the last dimension of A must exactly equal the second-to-last dimension of B. The resulting matrix will have the shape (..., M, N).

torch.Size([2, 2, 2, 6])
2 inputs 2 no.of heads 2 head dimesion 6 tokens


In [98]:
mask = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1)
maskedAttention = torch.masked_fill(attentionScore,mask=mask.bool(), value=-torch.inf)
maskedAttentionWeight = torch.softmax(maskedAttention/key.shape[-1]**0.5, dim=-1)
dropOutLayer = nn.Dropout(0.5)
maskedAttentionWeight_dropOut = dropOutLayer(maskedAttentionWeight)
print("maskedAttentionWeight_dropOut",maskedAttentionWeight_dropOut.shape)

maskedAttentionWeight_dropOut torch.Size([2, 2, 6, 6])


In [99]:
print("maskedAttentionWeight_dropOut", maskedAttentionWeight_dropOut.shape, "value", value.shape)
context_vectors = maskedAttentionWeight_dropOut @ value
print("context_vectors", context_vectors.shape)
context_vectorsTranspose = context_vectors.transpose(1,2)
print("context_vectorsTranspose", context_vectorsTranspose.shape)
print(context_vectorsTranspose.shape[0],"inputs", context_vectorsTranspose.shape[1], "tokens", context_vectorsTranspose.shape[2], "no.of heads", context_vectorsTranspose.shape[3],"head dimesion")
#same as initial Tensor, back to original dimension set
context_vecFinal = context_vectorsTranspose.contiguous().view(batch_num, num_tokens, embed_dim)
print("context_vecFinal",context_vecFinal.shape)

maskedAttentionWeight_dropOut torch.Size([2, 2, 6, 6]) value torch.Size([2, 2, 6, 2])
context_vectors torch.Size([2, 2, 6, 2])
context_vectorsTranspose torch.Size([2, 6, 2, 2])
2 inputs 6 tokens 2 no.of heads 2 head dimesion
context_vecFinal torch.Size([2, 6, 4])


In [100]:
outProjection = nn.Linear(embed_dim,embed_dim)
contextFinalProjected = outProjection(context_vecFinal)
print("contextFinalProjected",contextFinalProjected.shape)

contextFinalProjected torch.Size([2, 6, 4])


In [107]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_tokens, dim_in, dim_out, batch_num, qkvBias, num_heads, head_dim, dropPValue):
        super().__init__()
        self.dim_out = dim_out
        self.head_dim = head_dim
        self.num_tokens = num_tokens
        self.num_heads = num_heads
        self.batch_num = batch_num
        self.queryWeightFunc = nn.Linear(dim_in,dim_out, bias=qkvBias)
        self.keyWeightFunc = nn.Linear(dim_in,dim_out, bias=qkvBias)
        self.valueWeightFunc = nn.Linear(dim_in,dim_out, bias=qkvBias)
        self.mask = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1)
        self.dropOut = nn.Dropout(p=dropPValue)
        self.outProjection = nn.Linear(dim_out,dim_out)

    def forward(self, batch):
        query = self.queryWeightFunc(batch)
        key = self.keyWeightFunc(batch)
        value = self.valueWeightFunc(batch)

        #splitting embedding into num_heads for each token across batches
        query = query.view(self.batch_num, self.num_tokens, self.num_heads, self.head_dim)
        key = key.view(self.batch_num, self.num_tokens, self.num_heads, self.head_dim)
        value = value.view(self.batch_num, self.num_tokens, self.num_heads, self.head_dim)

        #changing to dimension --- batch_num -> no.of heads --> no.of tokens --> embedding
        query = query.transpose(1,2)
        key = key.transpose(1,2)
        value = value.transpose(1,2)

        attentionScore = query @ key.transpose(2,3) # transposing last 2 dimensions for sake of multipciation rule
        attenScoreMasked = torch.masked_fill(attentionScore, mask=self.mask.bool(), value=-torch.inf)
        attentionWeight = torch.softmax(attenScoreMasked/key.shape[-1]**0.5, dim=-1)
        attentionWeight_dropOut = self.dropOut(attentionWeight)

        context_vectors = attentionWeight_dropOut @ value
        context_vectorsTranspose = context_vectors.transpose(1,2)
        context_vecFinal = context_vectorsTranspose.contiguous().view(self.batch_num, self.num_tokens, self.dim_out)
        return self.outProjection(context_vecFinal)

In [108]:
torch.manual_seed(123)
dim_in = embed_dim
dim_out = embed_dim #if you want you can reduce the dimension also -- just check if its compatible with number of heads
mha = MultiHeadAttention(num_tokens, dim_in, dim_out, batch_num, qkvBias=False, num_heads=2, head_dim=2, dropPValue=0.5)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.0913, -0.0889, -1.1297, -0.6064],
         [-0.1954,  0.0383, -0.0965, -0.0626],
         [-0.1426, -0.0334, -0.5911, -0.3021],
         [-0.1438, -0.0157, -0.4508, -0.2706],
         [-0.1749,  0.0089, -0.1975, -0.1243],
         [-0.0941, -0.0460, -0.6764, -0.3751]],

        [[-0.1991,  0.0201, -0.1166, -0.0549],
         [-0.1617, -0.0222, -0.7040, -0.2259],
         [-0.1781, -0.0493, -0.6383, -0.3225],
         [-0.1564, -0.0384, -0.6888, -0.2949],
         [-0.1192, -0.0210, -0.5887, -0.2561],
         [-0.1382, -0.0034, -0.3085, -0.1895]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
